## Regression Models

In [14]:
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
df= pd.read_csv("../data/student_dataset/student-mat.csv", sep=";")
df.head(5)
target="G3"
X=df.drop(columns=[target])
y=df[target]
print(y)

0       6
1       6
2      10
3      15
4      10
       ..
390     9
391    16
392     7
393    10
394     9
Name: G3, Length: 395, dtype: int64


In [15]:
# Train Test
X_train, X_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
numerical_cols=X.select_dtypes(include="number").columns
categorical_cols=X.select_dtypes(include=["object", "category"]).columns
numerical_transformer=StandardScaler()
categorical_cols_transformer=OneHotEncoder()
preprocessor=ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_cols),
        ("cat", categorical_cols_transformer, categorical_cols)
    ]
)

C:\Users\anamo\AppData\Local\Temp\ipykernel_11936\3481029041.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols=X.select_dtypes(include=["object", "category"]).columns


In [16]:
#DummyRegressor
baseline=DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
y_pred=baseline.predict(X_test)
print(y_pred)

[10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937 10.32594937
 10.32594937]


In [18]:
#Train and compare models
models={
    "linear":Pipeline([
        ("preproccessor", preprocessor),
        ("linear", LinearRegression())
    ]),
    "ridge":Pipeline([
        ("preprocessor", preprocessor),
        ("ridge", Ridge(alpha=1))
    ]),
    "lasso":Pipeline([
        ("preprocessor", preprocessor),
        ("lasso", Lasso(alpha=0.1, max_iter=100000))
    ]),
    "elasticnet":Pipeline([
        ("preprocessor", preprocessor),
        ("elasticnet", ElasticNet(alpha=1, l1_ratio=0.5, max_iter=1000))
    ])
}
results=[]
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred=model.predict(X_test)
    mae= mean_absolute_error(y_test, y_pred)
    rmse= mean_squared_error(y_test, y_pred) ** 0.5
    r2= r2_score(y_test, y_pred)
    coefficient=model.named_steps[name].coef_
    results.append({
        "mae":mae,
        "rmse":rmse,
        "r2": r2,
        "coefficient": coefficient
    })

results_df=pd.DataFrame(results)
results_df

,mae,rmse,r2,coefficient
0,1.646666,2.378370,0.724134,"[-0.14701881433152114, 0.09310677203525475, -0..."
1,1.639067,2.371532,0.725718,"[-0.15112744908233244, 0.09302112462789296, -0..."
2,1.281796,2.092800,0.786403,"[-0.13399381837722782, 0.0, -0.0, 0.0021515099..."
3,1.552775,2.331277,0.734951,"[-0.0, 0.0, 0.0, -0.0, 0.0, -0.160982798383261..."
